In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import SGDRegressor, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import mlflow
from mlflow.models import infer_signature
import warnings
warnings.filterwarnings('ignore')

In [3]:
import urllib.request
import zipfile
import os

zip_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip"
zip_path = "student.zip"

if not os.path.exists("student-mat.csv"):
    urllib.request.urlretrieve(zip_url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")

df = pd.read_csv("student-mat.csv", delimiter=";")
print(f"Shape: {df.shape}")
df.head()

Shape: (395, 33)


,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,4,3,4,1,1,3,6,5,6,6
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,5,3,3,1,1,3,4,5,5,6
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,4,3,2,2,3,3,10,7,8,10
3,GP,F,15,U,GT3,T,4,2,health,services,...,3,2,2,1,1,5,2,15,14,15
4,GP,F,16,U,GT3,T,3,3,other,other,...,4,3,2,1,2,5,4,6,10,10


In [4]:
def preprocess(frame):
    data = frame.copy()
    data = data.drop_duplicates().dropna().reset_index(drop=True)

    cat_columns = data.select_dtypes(include=["object"]).columns.tolist()

    if cat_columns:
        encoder = OrdinalEncoder()
        data[cat_columns] = encoder.fit_transform(data[cat_columns])

    q_low = data["G3"].quantile(0.01)
    q_high = data["G3"].quantile(0.99)
    data = data[(data["G3"] >= q_low) & (data["G3"] <= q_high)]
    data = data.reset_index(drop=True)

    return data

df_clean = preprocess(df)

X = df_clean.drop(columns=["G3"])
y = df_clean["G3"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y.values, test_size=0.3, random_state=42
)

print(f"Clean shape: {df_clean.shape}")
print(f"Features: {X.shape[1]}")
print(f"Train: {X_train.shape}, Val: {X_val.shape}")


Clean shape: (394, 33)
Features: 32
Train: (275, 32), Val: (119, 32)


In [5]:
def eval_metrics(actual, pred):
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mae = mean_absolute_error(actual, pred)
    r2 = r2_score(actual, pred)
    return rmse, mae, r2


In [6]:
mlflow.set_experiment("student_performance_lab3")

2026/04/09 05:18:54 INFO mlflow.tracking.fluent: Experiment with name 'student_performance_lab3' does not exist. Creating a new experiment.


<Experiment: artifact_location='/home/qwart/mlops_lab3/mlruns/1', creation_time=1775693934623, experiment_id='1', last_update_time=1775693934623, lifecycle_stage='active', name='student_performance_lab3', tags={}, trace_location=None, workspace='default'>

In [7]:
with mlflow.start_run(run_name="SGD_GridSearch_basic"):
    params = {
        'alpha': [0.0001, 0.001, 0.01, 0.05, 0.1],
        'l1_ratio': [0.001, 0.01, 0.05, 0.2]
    }
    lr = SGDRegressor(random_state=42, max_iter=1000)
    clf = GridSearchCV(lr, params, cv=5, scoring='r2')
    clf.fit(X_train, y_train)

    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    rmse, mae, r2 = eval_metrics(y_val, y_pred)

    mlflow.log_param("model_type", "SGDRegressor")
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("l1_ratio", best.l1_ratio)
    mlflow.log_param("penalty", best.penalty)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train, best.predict(X_train))
    mlflow.sklearn.log_model(best, "model", signature=signature)

    print(f"[Exp 1] Best params: {clf.best_params_}")
    print(f"[Exp 1] RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")


2026/04/09 05:19:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 05:19:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Exp 1] Best params: {'alpha': 0.05, 'l1_ratio': 0.001}
[Exp 1] RMSE=2.1632, MAE=1.5094, R2=0.7581


In [8]:
with mlflow.start_run(run_name="SGD_GridSearch_extended"):
    params = {
        'alpha': [0.0001, 0.001, 0.01],
        'penalty': ['l1', 'l2', 'elasticnet'],
        'loss': ['squared_error', 'huber', 'epsilon_insensitive'],
        'l1_ratio': [0.01, 0.15],
        'fit_intercept': [True, False]
    }
    lr = SGDRegressor(random_state=42, max_iter=2000)
    clf = GridSearchCV(lr, params, cv=5, scoring='r2', n_jobs=-1)
    clf.fit(X_train, y_train)

    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    rmse, mae, r2 = eval_metrics(y_val, y_pred)

    mlflow.log_param("model_type", "SGDRegressor_extended")
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("l1_ratio", best.l1_ratio)
    mlflow.log_param("penalty", best.penalty)
    mlflow.log_param("loss", best.loss)
    mlflow.log_param("fit_intercept", best.fit_intercept)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train, best.predict(X_train))
    mlflow.sklearn.log_model(best, "model", signature=signature)

    print(f"[Exp 2] Best params: {clf.best_params_}")
    print(f"[Exp 2] RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")


2026/04/09 05:19:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 05:19:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Exp 2] Best params: {'alpha': 0.0001, 'fit_intercept': True, 'l1_ratio': 0.01, 'loss': 'epsilon_insensitive', 'penalty': 'l1'}
[Exp 2] RMSE=2.0344, MAE=1.2097, R2=0.7860


In [9]:
with mlflow.start_run(run_name="Ridge_GridSearch"):
    params = {
        'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
        'fit_intercept': [True, False]
    }
    ridge = Ridge(random_state=42)
    clf = GridSearchCV(ridge, params, cv=5, scoring='r2')
    clf.fit(X_train, y_train)

    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    rmse, mae, r2 = eval_metrics(y_val, y_pred)

    mlflow.log_param("model_type", "Ridge")
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("fit_intercept", best.fit_intercept)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train, best.predict(X_train))
    mlflow.sklearn.log_model(best, "model", signature=signature)

    print(f"[Exp 3] Best params: {clf.best_params_}")
    print(f"[Exp 3] RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")


2026/04/09 05:20:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 05:20:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Exp 3] Best params: {'alpha': 10.0, 'fit_intercept': True}
[Exp 3] RMSE=2.1623, MAE=1.5010, R2=0.7583


In [10]:
with mlflow.start_run(run_name="Lasso_GridSearch"):
    params = {
        'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
        'fit_intercept': [True, False]
    }
    lasso = Lasso(random_state=42, max_iter=5000)
    clf = GridSearchCV(lasso, params, cv=5, scoring='r2')
    clf.fit(X_train, y_train)

    best = clf.best_estimator_
    y_pred = best.predict(X_val)
    rmse, mae, r2 = eval_metrics(y_val, y_pred)

    mlflow.log_param("model_type", "Lasso")
    mlflow.log_param("alpha", best.alpha)
    mlflow.log_param("fit_intercept", best.fit_intercept)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    signature = infer_signature(X_train, best.predict(X_train))
    mlflow.sklearn.log_model(best, "model", signature=signature)

    print(f"[Exp 4] Best params: {clf.best_params_}")
    print(f"[Exp 4] RMSE={rmse:.4f}, MAE={mae:.4f}, R2={r2:.4f}")


2026/04/09 05:20:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/09 05:20:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


[Exp 4] Best params: {'alpha': 0.1, 'fit_intercept': True}
[Exp 4] RMSE=2.0176, MAE=1.3008, R2=0.7895


In [11]:
runs_df = mlflow.search_runs(experiment_names=["student_performance_lab3"])
comparison = runs_df[[
    "tags.mlflow.runName",
    "params.model_type",
    "params.alpha",
    "metrics.r2",
    "metrics.rmse",
    "metrics.mae"
]].sort_values("metrics.r2", ascending=False)

comparison


,tags.mlflow.runName,params.model_type,params.alpha,metrics.r2,metrics.rmse,metrics.mae
0,Lasso_GridSearch,Lasso,0.1,0.789537,2.017638,1.300842
2,SGD_GridSearch_extended,SGDRegressor_extended,0.0001,0.786021,2.034421,1.209699
1,Ridge_GridSearch,Ridge,10.0,0.758277,2.162294,1.501042
3,SGD_GridSearch_basic,SGDRegressor,0.05,0.758069,2.163222,1.509362


In [12]:
best_run = runs_df.sort_values("metrics.r2", ascending=False).iloc[0]
best_run_id = best_run["run_id"]
model_uri = f"runs:/{best_run_id}/model"

print(f"Best run: {best_run['tags.mlflow.runName']}")
print(f"Model:    {best_run['params.model_type']}")
print(f"R2:       {best_run['metrics.r2']:.4f}")
print(f"RMSE:     {best_run['metrics.rmse']:.4f}")
print(f"MAE:      {best_run['metrics.mae']:.4f}")
print(f"Run ID:   {best_run_id}")
print(f"URI:      {model_uri}")


Best run: Lasso_GridSearch
Model:    Lasso
R2:       0.7895
RMSE:     2.0176
MAE:      1.3008
Run ID:   2defa2337f114c868db0e3aabef15c29
URI:      runs:/2defa2337f114c868db0e3aabef15c29/model


In [13]:
loaded_model = mlflow.sklearn.load_model(model_uri)

test_pred = loaded_model.predict(X_val[:5])
print("Predictions:", test_pred)
print("Actual:     ", y_val[:5])


Predictions: [ 4.23129151 11.37754241  3.6651803  15.06330917  8.86154947]
Actual:      [ 5  9  5 15  9]


In [15]:
print(best_run_id)

2defa2337f114c868db0e3aabef15c29


In [16]:
import requests

url = "http://127.0.0.1:5001/invocations"
data = {"inputs": [X_val[0].tolist()]}
response = requests.post(
    url, json=data,
    headers={"Content-Type": "application/json"}
)

print(f"Prediction: {response.json()}")
print(f"Actual:     {y_val[0]}")


Prediction: {'predictions': [4.231291510627612]}
Actual:     5


In [17]:
print(X_val[0].tolist())


[-0.3635710495157405, -0.9456377579341728, -0.5471870284976068, 0.5362664443598957, -0.6380774695464941, 0.340803915443866, 0.2322293075604723, 1.3600981322540622, -1.7720174771096384, -0.3235910006882159, -1.0383734769721082, 0.2743183130319777, -0.6444037141900663, -0.03654666015206699, -0.4506300656443758, -0.38560089036104456, 0.792527080643759, -0.921827130331909, -1.0153461651336189, 0.5087100770497993, 0.23124864503144038, 0.4485749794573053, -0.7098005310600505, -2.168241026595867, 0.7637767599710122, -0.10043882207733545, -0.5415853578709906, -0.22889526263595733, -0.4022616830696411, 0.7855918483866062, -1.7875880024493203, -1.5228929120440609]
